In [ ]:
## The Curvilinear Transformation is used for I-294L1, I-294L1 and I-9094 (since I-395 is already parallel to y-axis)
## (you can use other tgsim datasets for your practice)
### main code here  (find centerline and save the Curvilinear Transformation of the datasets )

import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

_NGM_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _NGM_ROOT not in sys.path:
    sys.path.insert(0, _NGM_ROOT)
from ngm_paths import calibration_dataset_paths, lateral_processed_dir

x = "xloc_kf"
y = "yloc_kf"

datasets = {
    "df294l1": calibration_dataset_paths()["I294l1"],
    "df294l2": calibration_dataset_paths()["I294l2"],
    "df9094": calibration_dataset_paths()["I9094"],
}
PROCESSED_DIR = lateral_processed_dir()
os.makedirs(PROCESSED_DIR, exist_ok=True)

for dataset_name, file_path in datasets.items():
    # Check if the file exists
    if not os.path.exists(file_path):
        print(f"Error: The file '{file_path}' was not found. Skipping...")
        continue

    # Load the dataset
    data = pd.read_csv(file_path)

    # Ensure there is a run-index column (needed for all processing)
    if "run_index" not in data.columns:
        data["run_index"] = 1

    # Process each run-index separately
    unique_run_indices = np.sort(data['run_index'].unique())
    
    # Remove run 8 for df294l1
    if dataset_name == "df294l1":
        unique_run_indices = unique_run_indices[unique_run_indices != 8]
    
    for run_index in unique_run_indices:
        print(dataset_name, run_index)
        run_data = data[data['run_index'] == run_index].copy()

        # Extract unique lanes
        unique_lanes = np.sort(run_data['lane_kf'].unique())

        # Plot before transformation (508 compliant: no color encoding, no point markers)
        plt.figure(figsize=(10, 5))
        line_cycle = ['-', '--', '-.', ':']
        lane_styles = {}
        for idx, lane_id in enumerate(unique_lanes):
            line_style = line_cycle[idx % len(line_cycle)]
            lane_styles[lane_id] = line_style

        plotted_lanes = set()
        for vehicle_id in sorted(run_data['id'].unique()):
            vehicle_data = run_data[run_data['id'] == vehicle_id].sort_values(x)
            lane_id = vehicle_data['lane_kf'].mode().iloc[0]
            line_style = lane_styles[lane_id]
            label = f'Lane {lane_id}' if lane_id not in plotted_lanes else None
            plotted_lanes.add(lane_id)
            plt.plot(
                vehicle_data[x],
                vehicle_data[y],
                linestyle=line_style,
                color='black',
                linewidth=1.2,
                alpha=0.9,
                label=label,
            )
        plt.xlabel(x)
        plt.ylabel(y)
        plt.title(f'Original Road with Centerline - {dataset_name} (Run {run_index})')
        plt.legend()
        plt.grid()
        plt.show()

        # Use the lowest-numbered lane as the reference centerline
        reference_lane = unique_lanes.min()
        reference_data = run_data[run_data['lane_kf'] == reference_lane]
        reference_x = reference_data[x].values
        reference_y = reference_data[y].values
        lowess_reference = lowess(reference_y, reference_x, frac=0.03, return_sorted=True)
        x_reference = lowess_reference[:, 0]
        y_reference = lowess_reference[:, 1]

        # Process each lane separately with vectorized operations
        lane_offsets = {}
        all_x = run_data[x].values
        all_y = run_data[y].values
        all_lanes = run_data['lane_kf'].values

        for lane_id in unique_lanes:
            lane_mask = all_lanes == lane_id
            x_lane = all_x[lane_mask]
            y_lane = all_y[lane_mask]

            # Fit LOWESS curve for the lane
            lowess_result = lowess(y_lane, x_lane, frac=0.03, return_sorted=True)
            x_center = lowess_result[:, 0]
            y_center = lowess_result[:, 1]

            # Compute lane offset using reference centerline
            y_center_interp_ref = np.interp(x_center, x_reference, y_reference)
            lane_offset = np.mean(y_center - y_center_interp_ref)
            lane_offsets[lane_id] = lane_offset

            # Apply transformation using vectorized operations
            y_center_interp = np.interp(x_lane, x_center, y_center)
            all_y[lane_mask] = (y_lane - y_center_interp) + lane_offset

        # Assign transformed y-values back to run_data
        run_data[y] = all_y
        data.loc[data['run_index'] == run_index, y] = run_data[y]
        
        # Plot after transformation (508 compliant: no color encoding, no point markers)
        plt.figure(figsize=(10, 5))
        line_cycle = ['-', '--', '-.', ':']
        lane_styles = {}
        for idx, lane_id in enumerate(unique_lanes):
            line_style = line_cycle[idx % len(line_cycle)]
            lane_styles[lane_id] = line_style

        plotted_lanes = set()
        for vehicle_id in sorted(run_data['id'].unique()):
            vehicle_data = run_data[run_data['id'] == vehicle_id].sort_values(x)
            lane_id = vehicle_data['lane_kf'].mode().iloc[0]
            line_style = lane_styles[lane_id]
            label = f'Lane {lane_id}' if lane_id not in plotted_lanes else None
            plotted_lanes.add(lane_id)
            plt.plot(
                vehicle_data[x],
                vehicle_data[y],
                linestyle=line_style,
                color='black',
                linewidth=1.2,
                alpha=0.9,
                label=label,
            )
        plt.xlabel('Aligned X')
        plt.ylabel('Relative Y to Centerline with Offset')
        plt.title(f'Straightened Road (Relative to Reference Centerline) - {dataset_name} (Run {run_index})')
        plt.legend()
        plt.grid()
        plt.show() #(Comment out for plotting)

    # Save transformed data
    input_path = datasets[dataset_name]
    original_filename = os.path.basename(input_path)
    name_without_ext = os.path.splitext(original_filename)[0]
    output_filename = f"{name_without_ext}_Curvilinear_Transformation_for_Lateral_Calibration.csv"
    output_path = os.path.join(PROCESSED_DIR, output_filename)
    # Save CSV
    data.to_csv(output_path, index=False)
    print(f"Transformation data saved to {output_path}")